# Module 2.3 — Filtering, Joining, and Summarising Data

## Answering questions with fictional appointment records

A clean table is useful, but analysis begins when we ask focused questions of it. In this lesson, we will filter rows, sort results, calculate grouped summaries, combine related tables, and reshape results into readable reports.

> **Privacy note:** Every record is fictional and contains no real patient information. The examples concern appointment administration, not clinical diagnosis or treatment.

## Learning objectives

By the end of this notebook, you will be able to:

- Translate a plain-language question into a pandas operation.
- Filter rows with Boolean conditions, `isin()`, `between()`, and missing-value checks.
- Combine conditions safely with `&`, `|`, and `~`.
- Select columns with `.loc[]` and order results with `sort_values()`.
- Distinguish `size()`, `count()`, and `nunique()`.
- Use `groupby()` and named aggregation to calculate group-level summaries.
- Join appointment records to a department lookup table.
- Explain inner, left, right, and outer joins.
- Validate join cardinality and identify unmatched keys.
- Build frequency tables with `crosstab()` and summary matrices with `pivot_table()`.
- Communicate findings without making unsupported causal claims.

## Lesson roadmap

| Part | Focus |
|---|---|
| 1 | Prepare the fictional tables |
| 2 | Filter and sort records |
| 3 | Group and aggregate |
| 4 | Join related tables safely |
| 5 | Build cross-tabulations and pivot tables |
| 6 | Answer operational questions |
| 7 | Practice and test your knowledge |

**Working pattern:** state the question → identify the required rows and columns → calculate → validate → interpret.

## 1. Setup

This notebook requires pandas. If the import fails, install it in the environment used by this notebook:

```text
python -m pip install pandas
```

In [ ]:
import pandas as pd

print("pandas version:", pd.__version__)

## 2. Create the appointments table

Each row represents one fictional scheduled appointment. The data is already in a simple cleaned form so we can concentrate on selecting, combining, and summarising.

The values are invented only to demonstrate pandas operations. They do not describe real clinic performance.

In [ ]:
appointments = [
    ["APT-2001", "2026-09-01", "D01", "Initial",   "Completed", 12, 20, True,  "Phone"],
    ["APT-2002", "2026-09-01", "D02", "Follow-up", "Completed", 25, 30, True,  "Online"],
    ["APT-2003", "2026-09-02", "D03", "Initial",   "Completed",  8, 20, False, "Phone"],
    ["APT-2004", "2026-09-02", "D01", "Follow-up", "Cancelled", pd.NA, 15, True, "Online"],
    ["APT-2005", "2026-09-03", "D02", "Initial",   "Completed", 40, 45, True,  "Reception"],
    ["APT-2006", "2026-09-03", "D01", "Initial",   "Missed",   15, 20, False, "Phone"],
    ["APT-2007", "2026-09-04", "D03", "Follow-up", "Completed", 18, 15, True,  "Online"],
    ["APT-2008", "2026-09-04", "D01", "Initial",   "Completed",  6, 20, True,  "Reception"],
    ["APT-2009", "2026-09-05", "D02", "Follow-up", "Completed", 22, 30, True,  "Online"],
    ["APT-2010", "2026-09-05", "D03", "Initial",   "Cancelled", pd.NA, 20, False, "Phone"],
    ["APT-2011", "2026-09-06", "D01", "Follow-up", "Completed", 10, 15, True,  "Online"],
    ["APT-2012", "2026-09-06", "D02", "Initial",   "Missed",   30, 45, True,  "Reception"],
    ["APT-2013", "2026-09-07", "D03", "Initial",   "Completed", 14, 20, True,  "Online"],
    ["APT-2014", "2026-09-07", "D01", "Follow-up", "Completed", 16, 15, False, "Phone"],
    ["APT-2015", "2026-09-08", "D02", "Initial",   "Cancelled", pd.NA, 45, True, "Online"],
    ["APT-2016", "2026-09-08", "D03", "Follow-up", "Missed",   20, 15, False, "Reception"],
    ["APT-2017", "2026-09-09", "D01", "Initial",   "Completed",  9, 20, True,  "Online"],
    ["APT-2018", "2026-09-09", "D02", "Follow-up", "Completed", 28, 30, True,  "Phone"],
    ["APT-2019", "2026-09-10", "D03", "Initial",   "Completed", 11, 20, True,  "Reception"],
    ["APT-2020", "2026-09-10", "D99", "Initial",   "Completed", 17, 25, True,  "Online"],
]

columns = [
    "appointment_id", "appointment_date", "department_id", "appointment_type",
    "status", "wait_minutes", "scheduled_minutes", "reminder_sent", "booking_channel"
]

appointments_df = pd.DataFrame(appointments, columns=columns)
appointments_df["appointment_date"] = pd.to_datetime(appointments_df["appointment_date"])
appointments_df["wait_minutes"] = appointments_df["wait_minutes"].astype("Int64")
appointments_df["scheduled_minutes"] = appointments_df["scheduled_minutes"].astype("Int64")
appointments_df.head()

### Inspect the starting table

Before answering questions, verify the unit of observation, dimensions, types, missing values, and identifier uniqueness.

In [ ]:
print("Shape:", appointments_df.shape)
print("Duplicate appointment IDs:", appointments_df["appointment_id"].duplicated().sum())
print("\nMissing values:")
print(appointments_df.isna().sum())
print("\nData types:")
print(appointments_df.dtypes)

## 3. Create a department lookup table

The appointment table stores a short `department_id`. A separate lookup table stores descriptive department information.

This design avoids repeating the same department name, floor, and capacity in every appointment row.

In [ ]:
departments_df = pd.DataFrame({
    "department_id": ["D01", "D02", "D03", "D04"],
    "department_name": [
        "General Practice", "Physiotherapy", "Dermatology", "Imaging"
    ],
    "floor": [1, 2, 2, 1],
    "daily_capacity": [18, 12, 10, 8],
})

departments_df

### Predict before joining

Compare the two key columns:

- Which appointment key has no matching lookup row?
- Which department appears in the lookup but has no appointments?
- Should joining department information increase the number of appointment rows?

Keep your answers in mind. We will test them later.

## 4. Boolean filtering

A Boolean condition produces `True` or `False` for each row. Applying that mask keeps rows where the condition is `True`.

In [ ]:
completed_mask = appointments_df["status"] == "Completed"
completed_mask.head()

In [ ]:
completed_appointments = appointments_df.loc[completed_mask]
completed_appointments.head()

> Read the condition aloud: “Keep rows where status equals Completed.” This habit makes complex filters easier to check.

Filtering creates a subset; it does not alter the original table unless we assign changes back to it.

In [ ]:
print("All appointments:", len(appointments_df))
print("Completed appointments:", len(completed_appointments))

## 5. Select rows and columns together

`.loc[row_condition, column_list]` makes both decisions explicit.

In [ ]:
appointments_df.loc[
    appointments_df["status"] == "Missed",
    ["appointment_id", "appointment_date", "department_id", "reminder_sent"],
]

## 6. Comparison filters

Use `>`, `>=`, `<`, and `<=` for numerical or date comparisons. The next query keeps recorded waiting times of at least 20 minutes.

In [ ]:
long_waits = appointments_df.loc[
    appointments_df["wait_minutes"] >= 20,
    ["appointment_id", "department_id", "wait_minutes", "status"],
]

long_waits

### Filter a date range

`between()` includes both endpoints by default.

In [ ]:
date_subset = appointments_df.loc[
    appointments_df["appointment_date"].between("2026-09-04", "2026-09-07"),
    ["appointment_id", "appointment_date", "status"],
]

date_subset

## 7. Missing-value filters

Use `isna()` and `notna()` rather than comparing a value to `None` or `pd.NA` with `==`.

In [ ]:
appointments_df.loc[
    appointments_df["wait_minutes"].isna(),
    ["appointment_id", "status", "wait_minutes"],
]

In [ ]:
recorded_waits = appointments_df.loc[
    appointments_df["wait_minutes"].notna(),
    ["appointment_id", "wait_minutes"],
]

print("Rows with a recorded wait:", len(recorded_waits))

## 8. Membership with `isin()`

`isin()` is useful when several allowed values belong to the same condition.

In [ ]:
not_completed = appointments_df.loc[
    appointments_df["status"].isin(["Cancelled", "Missed"]),
    ["appointment_id", "status", "department_id"],
]

not_completed

## 9. Combine conditions

In pandas Boolean filters:

- `&` means AND.
- `|` means OR.
- `~` means NOT.

Put each condition inside parentheses. Python's words `and` and `or` do not combine entire pandas Series element by element.

In [ ]:
# Completed AND waited at least 20 minutes
completed_long_wait = appointments_df.loc[
    (appointments_df["status"] == "Completed")
    & (appointments_df["wait_minutes"] >= 20),
    ["appointment_id", "department_id", "wait_minutes", "status"],
]

completed_long_wait

In [ ]:
# Booked online OR by reception
online_or_reception = appointments_df.loc[
    appointments_df["booking_channel"].isin(["Online", "Reception"]),
    ["appointment_id", "booking_channel", "status"],
]

online_or_reception.head()

In [ ]:
# NOT completed
not_completed_with_tilde = appointments_df.loc[
    ~(appointments_df["status"] == "Completed"),
    ["appointment_id", "status"],
]

not_completed_with_tilde

### Common filtering mistake

This expression is ambiguous and should not be used:

```python
appointments_df["status"] == "Completed" & appointments_df["wait_minutes"] >= 20
```

Use parentheses around each condition:

```python
(appointments_df["status"] == "Completed") & (appointments_df["wait_minutes"] >= 20)
```

## 10. Sort results

Sorting does not calculate a new metric; it changes row order. This helps us inspect high or low values.

In [ ]:
appointments_df.sort_values("wait_minutes", ascending=False)[
    ["appointment_id", "department_id", "wait_minutes", "status"]
].head(5)

### Sort by more than one column

The next result sorts departments alphabetically, then places the longest waits first within each department.

In [ ]:
appointments_df.sort_values(
    ["department_id", "wait_minutes"],
    ascending=[True, False],
)[["appointment_id", "department_id", "wait_minutes"]].head(10)

## 11. Filter checkpoint

Write filters for these questions:

1. Which appointments were completed and booked online?
2. Which appointments belong to departments `D01` or `D03`?
3. Which records have no waiting time?
4. Which completed records have a wait between 10 and 20 minutes?

In [ ]:
# Write your filters here.
# This comment-only cell allows the notebook to run from top to bottom.

<details>
<summary><strong>Reveal one possible solution</strong></summary>

Run the next cell after attempting the checkpoint.
</details>

In [ ]:
display(appointments_df.loc[
    (appointments_df["status"] == "Completed")
    & (appointments_df["booking_channel"] == "Online")
])

display(appointments_df.loc[
    appointments_df["department_id"].isin(["D01", "D03"])
])

display(appointments_df.loc[
    appointments_df["wait_minutes"].isna()
])

display(appointments_df.loc[
    (appointments_df["status"] == "Completed")
    & (appointments_df["wait_minutes"].between(10, 20))
])

## 12. From rows to groups

Filtering asks, “Which individual rows meet a condition?” Grouping asks, “How do summaries differ across categories?”

The `groupby()` pattern is often described as:

1. **Split** rows into groups.
2. **Apply** a calculation within each group.
3. **Combine** the results into a new object.

### Count rows with `size()`

`size()` counts rows in each group, including rows containing missing values elsewhere.

In [ ]:
appointments_by_status = (
    appointments_df
    .groupby("status")
    .size()
    .sort_values(ascending=False)
)

appointments_by_status

### `size()` versus `count()`

- `size()` counts rows.
- `count()` counts non-missing values in a selected column.

This difference matters because cancelled appointments have missing waiting times in our fictional data.

In [ ]:
status_counts = appointments_df.groupby("status").agg(
    rows=("appointment_id", "size"),
    recorded_waits=("wait_minutes", "count"),
)

status_counts

### Count distinct values with `nunique()`

`nunique()` answers a different question: how many distinct values appear?

In [ ]:
appointments_df.groupby("department_id")["booking_channel"].nunique()

## 13. Named aggregation

Named aggregation lets us choose clear output column names while applying different calculations.

In [ ]:
department_summary = (
    appointments_df
    .groupby("department_id", as_index=False)
    .agg(
        appointment_count=("appointment_id", "size"),
        completed_count=("status", lambda values: (values == "Completed").sum()),
        recorded_wait_count=("wait_minutes", "count"),
        mean_wait_minutes=("wait_minutes", "mean"),
        median_wait_minutes=("wait_minutes", "median"),
        total_scheduled_minutes=("scheduled_minutes", "sum"),
    )
)

department_summary[["mean_wait_minutes", "median_wait_minutes"]] = (
    department_summary[["mean_wait_minutes", "median_wait_minutes"]].round(1)
)

department_summary

> Missing waiting times are excluded from the mean and median. That is why the summary also reports `recorded_wait_count`. A mean without its contributing count can be misleading.

## 14. Group by two categories

Grouping by multiple columns creates one group for every observed combination.

In [ ]:
department_status_counts = (
    appointments_df
    .groupby(["department_id", "status"], as_index=False)
    .agg(appointment_count=("appointment_id", "size"))
)

department_status_counts

## 15. Create derived indicators

A Boolean column can be averaged because `True` behaves like 1 and `False` like 0. The mean is therefore a proportion.

We will create simple indicators for completed and missed appointments.

In [ ]:
analysis_df = appointments_df.copy()
analysis_df["is_completed"] = analysis_df["status"].eq("Completed")
analysis_df["is_missed"] = analysis_df["status"].eq("Missed")

outcome_summary = (
    analysis_df
    .groupby("department_id", as_index=False)
    .agg(
        appointments=("appointment_id", "size"),
        completion_rate=("is_completed", "mean"),
        missed_rate=("is_missed", "mean"),
    )
)

outcome_summary[["completion_rate", "missed_rate"]] = (
    outcome_summary[["completion_rate", "missed_rate"]].mul(100).round(1)
)

outcome_summary

> These percentages describe only the small fictional file. They do not establish why outcomes differ, and they should not be treated as stable estimates of department performance.

## 16. Grouping checkpoint

Create a summary by `booking_channel` containing:

- Number of appointments.
- Number completed.
- Mean recorded wait.
- Number of distinct departments.

Use readable output names and round the mean wait to one decimal place.

In [ ]:
# Build your booking-channel summary here.

<details>
<summary><strong>Reveal one possible solution</strong></summary>

Run the next cell after attempting the checkpoint.
</details>

In [ ]:
channel_summary = (
    appointments_df
    .groupby("booking_channel", as_index=False)
    .agg(
        appointment_count=("appointment_id", "size"),
        completed_count=("status", lambda values: (values == "Completed").sum()),
        mean_wait_minutes=("wait_minutes", "mean"),
        distinct_departments=("department_id", "nunique"),
    )
)

channel_summary["mean_wait_minutes"] = channel_summary["mean_wait_minutes"].round(1)
channel_summary

## 17. Why join tables?

The appointment table contains events. The department table contains descriptive information. A **join** combines them using a shared key.

The key question is not simply “Which command joins tables?” It is “Which rows should survive, and what relationship should exist between the keys?”

### Four common join types

| Join | Keeps |
|---|---|
| Inner | Keys found in both tables |
| Left | Every row from the left table, plus matches from the right |
| Right | Every row from the right table, plus matches from the left |
| Outer | Every key from either table |

For appointment analysis, a left join is often useful because it preserves all appointment records while revealing missing lookup matches.

## 18. Check keys before joining

The intended relationship is **many appointments to one department**. Therefore:

- Duplicate `department_id` values are expected in appointments.
- `department_id` should be unique in the department lookup.

In [ ]:
print("Appointment keys unique?", appointments_df["department_id"].is_unique)
print("Department lookup keys unique?", departments_df["department_id"].is_unique)
print("Duplicate lookup keys:", departments_df["department_id"].duplicated().sum())

## 19. Perform a validated left join

- `validate="many_to_one"` checks the expected key relationship.
- `indicator=True` adds a `_merge` column showing whether each key matched.

In [ ]:
appointments_enriched = appointments_df.merge(
    departments_df,
    how="left",
    on="department_id",
    validate="many_to_one",
    indicator=True,
)

appointments_enriched.head()

### Validate the join result

A many-to-one left join should preserve the number of left-table rows. We also inspect the merge indicator rather than assuming every key matched.

In [ ]:
print("Rows before join:", len(appointments_df))
print("Rows after join:", len(appointments_enriched))
print("\nMerge results:")
print(appointments_enriched["_merge"].value_counts())

assert len(appointments_enriched) == len(appointments_df)
print("\nRow-count assertion passed.")

In [ ]:
unmatched_appointments = appointments_enriched.loc[
    appointments_enriched["_merge"] == "left_only",
    ["appointment_id", "department_id"],
]

unmatched_appointments

> The unmatched `D99` record remains in the left-joined table, but its department details are missing. We should investigate or request a lookup correction rather than inventing a department name.

## 20. Compare join types

Run the same relationship with different `how` values and compare the key sets.

In [ ]:
inner_join = appointments_df.merge(
    departments_df,
    how="inner",
    on="department_id",
    validate="many_to_one",
)

outer_join = appointments_df.merge(
    departments_df,
    how="outer",
    on="department_id",
    indicator=True,
)

print("Left join rows:", len(appointments_enriched))
print("Inner join rows:", len(inner_join))
print("Outer join rows:", len(outer_join))
print("\nOuter join source counts:")
print(outer_join["_merge"].value_counts())

In [ ]:
outer_join.loc[
    outer_join["_merge"] != "both",
    ["appointment_id", "department_id", "department_name", "_merge"],
]

### Why row counts can unexpectedly grow

If the lookup table contains the same department key twice, each matching appointment may be repeated. The `validate` argument can stop the join instead of allowing silent row multiplication.

In [ ]:
bad_departments_df = pd.concat(
    [departments_df, departments_df.iloc[[0]]],
    ignore_index=True,
)

print("Duplicate lookup keys:", bad_departments_df["department_id"].duplicated().sum())

try:
    appointments_df.merge(
        bad_departments_df,
        how="left",
        on="department_id",
        validate="many_to_one",
    )
except pd.errors.MergeError as error:
    print("Merge stopped safely:", error)

## 21. Remove the merge indicator after checking

Once the match has been reviewed, we can remove `_merge` from the working table. We retain a separate unmatched-record table as evidence.

In [ ]:
appointments_enriched = appointments_enriched.drop(columns="_merge")
appointments_enriched.head(3)

## 22. Summarise after joining

The enriched table lets us group using readable department names and include lookup attributes such as floor and capacity.

In [ ]:
named_department_summary = (
    appointments_enriched
    .groupby(["department_id", "department_name"], dropna=False, as_index=False)
    .agg(
        appointment_count=("appointment_id", "size"),
        mean_wait_minutes=("wait_minutes", "mean"),
        total_scheduled_minutes=("scheduled_minutes", "sum"),
    )
)

named_department_summary["mean_wait_minutes"] = (
    named_department_summary["mean_wait_minutes"].round(1)
)

named_department_summary

> `dropna=False` keeps the unmatched `D99` group visible. Without it, a group with missing `department_name` could disappear from the summary.

## 23. Cross-tabulations

`pd.crosstab()` counts combinations of categories by default. Rows and columns make it easy to compare statuses across departments.

In [ ]:
status_crosstab = pd.crosstab(
    index=appointments_enriched["department_name"].fillna("Unmatched department"),
    columns=appointments_enriched["status"],
    margins=True,
    margins_name="Total",
)

status_crosstab

### Row percentages

Counts answer “How many?” Row percentages answer “What share of each department's appointments had each status?”

In [ ]:
status_row_percent = pd.crosstab(
    index=appointments_enriched["department_name"].fillna("Unmatched department"),
    columns=appointments_enriched["status"],
    normalize="index",
).mul(100).round(1)

status_row_percent

> Always label whether a table contains counts, row percentages, column percentages, or percentages of the whole dataset. The same values can answer very different questions depending on the denominator.

## 24. Pivot tables

A pivot table aggregates a numerical value across row and column categories. The next table calculates mean recorded waiting time by department and appointment type.

In [ ]:
wait_pivot = pd.pivot_table(
    data=appointments_enriched.assign(
        department_label=appointments_enriched["department_name"].fillna("Unmatched department")
    ),
    values="wait_minutes",
    index="department_label",
    columns="appointment_type",
    aggfunc="mean",
    margins=True,
    margins_name="Overall",
)

wait_pivot.round(1)

### Counts in a pivot table

Here we count appointment IDs and replace absent combinations with zero.

In [ ]:
count_pivot = pd.pivot_table(
    data=appointments_enriched.assign(
        department_label=appointments_enriched["department_name"].fillna("Unmatched department")
    ),
    values="appointment_id",
    index="department_label",
    columns="booking_channel",
    aggfunc="count",
    fill_value=0,
    margins=True,
    margins_name="Total",
)

count_pivot

## 25. Answer three operational questions

A useful analysis links each question to its method and reports limitations.

### Question 1: How many appointments are recorded for each department?

In [ ]:
question_1 = (
    appointments_enriched
    .assign(department_label=lambda df: df["department_name"].fillna("Unmatched department"))
    .groupby("department_label", as_index=False)
    .agg(appointment_count=("appointment_id", "size"))
    .sort_values("appointment_count", ascending=False)
)

question_1

**Interpretation:** The result describes counts in this fictional extract. It does not measure demand unless the extract contains all relevant appointments for a clearly defined period.

### Question 2: What is the mean recorded wait for completed appointments?

In [ ]:
question_2 = (
    appointments_enriched
    .loc[
        (appointments_enriched["status"] == "Completed")
        & appointments_enriched["wait_minutes"].notna()
    ]
    .assign(department_label=lambda df: df["department_name"].fillna("Unmatched department"))
    .groupby("department_label", as_index=False)
    .agg(
        completed_records=("appointment_id", "size"),
        mean_wait_minutes=("wait_minutes", "mean"),
        median_wait_minutes=("wait_minutes", "median"),
    )
)

question_2[["mean_wait_minutes", "median_wait_minutes"]] = (
    question_2[["mean_wait_minutes", "median_wait_minutes"]].round(1)
)
question_2

**Interpretation:** Reporting the contributing record count alongside the mean helps readers judge how much data underlies each value. The table does not explain why waits differ.

### Question 3: How are appointment statuses distributed within each department?

In [ ]:
question_3_counts = pd.crosstab(
    appointments_enriched["department_name"].fillna("Unmatched department"),
    appointments_enriched["status"],
)

question_3_percent = pd.crosstab(
    appointments_enriched["department_name"].fillna("Unmatched department"),
    appointments_enriched["status"],
    normalize="index",
).mul(100).round(1)

display(question_3_counts)
display(question_3_percent)

**Interpretation:** Counts show volume; row percentages show composition. With few fictional records, a single appointment can noticeably change a percentage.

## 26. Reusable question function

A function can apply one metric definition consistently. This function summarises only completed appointments with a recorded waiting time.

In [ ]:
def completed_wait_summary(df, group_column):
    """Summarise recorded waits for completed appointments by one column."""
    required = {group_column, "status", "wait_minutes", "appointment_id"}
    missing_columns = required.difference(df.columns)
    if missing_columns:
        raise KeyError(f"Missing columns: {sorted(missing_columns)}")

    subset = df.loc[
        (df["status"] == "Completed") & df["wait_minutes"].notna()
    ]

    result = (
        subset
        .groupby(group_column, dropna=False, as_index=False)
        .agg(
            appointment_count=("appointment_id", "size"),
            mean_wait_minutes=("wait_minutes", "mean"),
            median_wait_minutes=("wait_minutes", "median"),
        )
    )

    result[["mean_wait_minutes", "median_wait_minutes"]] = (
        result[["mean_wait_minutes", "median_wait_minutes"]].round(1)
    )
    return result


completed_wait_summary(appointments_enriched, "booking_channel")

## 27. Join and summary validation checklist

Before trusting a combined summary, check:

- The unit of observation in each table.
- The join key and its data type in both tables.
- Key uniqueness on the expected “one” side.
- The intended join type.
- Row counts before and after the join.
- Unmatched keys using a merge indicator.
- Missing values created by the join.
- Whether grouped calculations exclude missing values.
- Whether counts and denominators are shown with averages or rates.

## 28. Code-along challenge

Create a department report with these columns:

- `department_name`
- `floor`
- `appointment_count`
- `completed_count`
- `missed_count`
- `mean_recorded_wait`
- `total_scheduled_minutes`

Requirements:

1. Preserve the unmatched department as `Unmatched department`.
2. Use named aggregation.
3. Round the mean wait to one decimal place.
4. Sort by appointment count from largest to smallest.

In [ ]:
# Build your department report here.

<details>
<summary><strong>Reveal one possible solution</strong></summary>

Run the next cell after attempting the challenge.
</details>

In [ ]:
department_report = (
    appointments_enriched
    .assign(
        department_name=lambda df: df["department_name"].fillna("Unmatched department"),
        floor=lambda df: df["floor"].astype("Int64"),
    )
    .groupby(["department_name", "floor"], dropna=False, as_index=False)
    .agg(
        appointment_count=("appointment_id", "size"),
        completed_count=("status", lambda values: (values == "Completed").sum()),
        missed_count=("status", lambda values: (values == "Missed").sum()),
        mean_recorded_wait=("wait_minutes", "mean"),
        total_scheduled_minutes=("scheduled_minutes", "sum"),
    )
    .sort_values("appointment_count", ascending=False)
)

department_report["mean_recorded_wait"] = department_report["mean_recorded_wait"].round(1)
department_report

## 29. Independent practice

Use `appointments_enriched` to answer these questions:

1. Which five completed appointments have the longest recorded waits?
2. How many appointments came through each booking channel and status combination?
3. What percentage of appointments in each channel were missed?
4. What is the median recorded wait by department and appointment type?
5. Which lookup departments have no matching appointments?
6. Which appointment department IDs have no lookup match?

For each answer, add one sentence that states:

- What you observed.
- What the result does **not** prove.

In [ ]:
# Write your independent-practice analysis here.

### Suggested self-checks

- Did the five longest-wait records include only completed appointments?
- Does the channel/status table show counts rather than percentages?
- Is the denominator for the missed percentage clearly defined?
- Does the median pivot retain meaningful labels?
- Did you distinguish lookup-only keys from appointment-only keys?
- Did you avoid causal language such as “because” or “caused by”?

## 30. Stretch challenge: capacity context

The department lookup contains a fictional `daily_capacity`. Create a summary by appointment date and department that includes:

- Appointment count.
- Daily capacity.
- A simple `capacity_share_percent` calculated as appointment count divided by daily capacity.

Before interpreting the result, explain why this ratio is **not necessarily utilisation**. Consider whether the dataset is complete and what “capacity” means.

In [ ]:
daily_department_summary = (
    appointments_enriched
    .dropna(subset=["department_name", "daily_capacity"])
    .groupby(
        ["appointment_date", "department_name", "daily_capacity"],
        as_index=False,
    )
    .agg(appointment_count=("appointment_id", "size"))
)

daily_department_summary["capacity_share_percent"] = (
    daily_department_summary["appointment_count"]
    .div(daily_department_summary["daily_capacity"])
    .mul(100)
    .round(1)
)

daily_department_summary.head()

> This ratio compares recorded appointment count with the lookup value. Calling it utilisation would require evidence that the extract contains every relevant appointment and that `daily_capacity` represents comparable available slots.

## 31. Test your knowledge!

Answer without running new code first.

1. What does a Boolean mask contain?
2. Which operators represent AND, OR, and NOT in pandas filters?
3. Why should each condition be wrapped in parentheses?
4. What is the difference between `size()` and `count()`?
5. What does `nunique()` calculate?
6. What are the split, apply, and combine stages of `groupby()`?
7. Why can reporting a mean without a count be misleading?
8. Which join preserves every row from the left table?
9. In a many-to-one join, which table's key should be unique?
10. What information does `indicator=True` add to a merge?
11. Why should row counts be compared before and after a join?
12. When would a cross-tabulation be more suitable than a pivot table?
13. What denominator does `normalize="index"` use in `crosstab()`?
14. Why does a group-level difference not prove causation?

<details>
<summary><strong>Check your answers</strong></summary>

1. One `True` or `False` value for each row in the aligned data.
2. `&`, `|`, and `~`.
3. Parentheses make the intended comparison order explicit and prevent ambiguous expressions.
4. `size()` counts rows; `count()` counts non-missing values in the selected field.
5. The number of distinct non-missing values by default.
6. Divide records into groups, perform a calculation in each group, and combine the group results.
7. A mean based on very few non-missing records may appear equally reliable to one based on many records.
8. A left join.
9. The key on the “one” or lookup side.
10. A merge-source column showing left-only, right-only, or matched rows.
11. Unexpected growth can reveal duplicate keys and row multiplication; unexpected loss can reveal join-type or key problems.
12. When counting combinations of categorical variables.
13. Each row is its own denominator, so row percentages sum to 100% where categories are present.
14. Other variables, selection effects, data quality, or chance may explain the observed difference.

</details>

## 32. Lesson recap

You can now:

- Filter records using one or several conditions.
- Select relevant columns and sort results.
- Count rows, non-missing values, and distinct values correctly.
- Build readable group summaries with named aggregation.
- Combine event and lookup tables with an explicit join type.
- Validate key cardinality, row counts, and unmatched records.
- Use cross-tabulations for categorical counts and pivot tables for numerical summaries.
- Pair every operational result with its denominator, scope, and limitation.

In Module 2.4, we will explore distributions and relationships visually using Matplotlib and Seaborn.

## Further reading

- [pandas: Boolean indexing](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing)
- [pandas: Group by — split, apply, combine](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [pandas: Merge, join, concatenate, and compare](https://pandas.pydata.org/docs/user_guide/merging.html)
- [pandas: Reshaping and pivot tables](https://pandas.pydata.org/docs/user_guide/reshaping.html)
- [pandas: `crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)

Use the official documentation to confirm parameters, especially join validation and normalisation choices.